In [ ]:
import warnings
import os

import numpy as np 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer

warnings.simplefilter(action='ignore')

# Pima Indians Diabetes Study

Source: https://www.kaggle.com/uciml/pima-indians-diabetes-database

Pipeline of a machine learning study:

- loading data
- describing data, discrete/continuous, categorical/not categorical
- data cleaning (eliminating NaNs), converting text data to numeric values, removing duplicate rows
- basic distributions of features, correlation matrix
- data analysis with diagrams:
    - histograms
    - paired histograms
    - ratio for categorical data: pie chart
    - analysis along at least two dimensions: scatter plot
    - box plots

- data normalization
- training machine learning models
- evaluation of the results:
    - accuracy, confusion matrix in more dimension
    - ROCAUC curve
    - PPV, NPV (positive and negative predictive value, in binary classification)

<b> At each step, describe the current step with comment block (Markdown cell in Jupyter). While examining the data, try to come up with assumptions. In this notebook there some examples for that, these are for presentation purposes only, not medical facts </b>

In [ ]:
data_path = os.path.join("data", "diabetes.csv")
df = pd.read_csv(data_path)

In [ ]:
df.info()

In [ ]:
df.head(10)

In [ ]:
null_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

for col in null_cols:
    null_count = df[df[col]==0].shape[0]
    print(col, null_count, str(round(null_count/df.shape[0]*100, 1))+"%")

In [ ]:
# In these columns a 0 means "missing", not a real measurement. Replace the zeros
# with NaN first so they don't drag the median down, then fill with that median.
for col in null_cols:
    df[col] = df[col].replace(0, np.nan)
    df[col] = df[col].fillna(df[col].median())

df.head()

# Variables

### Categorical

Outcome - is diabetes daignosed? (1-yes, 0-no)

### Not categorical


Glucose - Blood glucose concentration

Insulin - Blood insulin concentration

SkinThickness - Triceps skin fold thickness (in mm)

BMI - Body-mass index

BloodPressure - Diastolic blood pressure (mm Hg)

DiabetesPedigreeFunction - A function which scores likelihood of diabetes based on family history

Pregnancies - Number of times pregnant

Age

# Plot distributions

In [ ]:
fig, ax = plt.subplots(figsize=(11, 11))
df.hist(bins=50, ax=ax)
plt.show()

# Correlation matrix

In this case the correlation matrix is applicable almost all cases, as the outcome is the only categorical feature. The strongest connection is between BMI and SkinThickness.

In [ ]:
sns.set_context("talk")
plt.figure(figsize=(15, 12))
mask = np.triu(df.corr())
sns.heatmap(df.corr(), annot=True, fmt=".2f",mask=mask)
plt.show()

# Age distribution

In [ ]:
sns.set_style("darkgrid")
sns.set_context("poster")
plt.figure(figsize=(12, 9))
sns.histplot(data=df["Age"], alpha=0.6)
plt.axvline(df["Age"].mean(), color="red")
plt.title("Distribution of age in the dataset")
plt.text(33, 42, "Mean age: " + str(round(df["Age"].mean(), 1)), color="indianred", weight="heavy", size="larger", bbox=dict(boxstyle="round", color="rosybrown"))
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()

# Average feature values grouped by Diabetes

In [ ]:
fig, axes = plt.subplots(figsize = (40, 30),nrows = 2, ncols = 4)
plot_these = ["Pregnancies", "Age", "Glucose", "Insulin", "BloodPressure", "SkinThickness", "BMI", "DiabetesPedigreeFunction" ]
for ax, cat in zip(axes.flat,plot_these):
    df.groupby("Outcome")[[cat]].mean().plot(kind="bar", ax = ax)
    ax.legend(loc='upper right')
plt.show()
df.groupby("Outcome")[plot_these].mean().plot(kind="bar")
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
df.groupby("Outcome")[plot_these].mean()

# Occurrence of Diabetes compared to glucose and insulin

In this graph, the zero values (missing data) would confuse the viewer in the insulin-axis, so it was replaced with the median of the sample. We can see a tendency of more positive cases with larger glucose levels, while they are somewhat evenly distributed along the horizontal axis (insulin), so the glucose value has higher importance in predicting the outcome.

In [ ]:


df["Test positivity"] = df["Outcome"].replace({1:"Positive", 0:"Negative"})

sns.set_style("darkgrid")
sns.set_context("poster")
plt.figure(figsize=(12, 9))
sns.scatterplot(data=df, x="Insulin", y="Glucose", hue=df["Test positivity"], )
plt.title("Cases presenting diabetes compared by their blood glucose and insulin levels")
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
plt.xlabel("Insulin level (mu U/mL)")
plt.ylabel("Glucose level")
plt.show()
help_df=df[['Outcome','Glucose','Insulin']]
pos_df=help_df.query('Outcome==1')
neg_df=help_df.query('Outcome==0')
Gl_values=[0,25,50,75,100,125,150,175,200]
In_values=[0,200,400,600,800]
pos_df["Insulin"]=pd.cut(pos_df.Insulin, In_values)
pos_df["Glucose"]=pd.cut(pos_df.Glucose, Gl_values)
display(pos_df.groupby(['Insulin', 'Glucose']).size())

# Comparing BMI and Age
From the plot it can be seen as almost all positive case has greater than 25 BMI. Also they occupy the 30-60 range in years, one possible explanation is that people with diabetes have shorter expected lifespan.

In [ ]:
sns.set_style("darkgrid")
sns.set_context("poster")
plt.figure(figsize=(12, 9))
sns.scatterplot(x="Age", y="BMI", hue=df["Test positivity"], data=df)
plt.title("Cases presenting diabetes compared by their BMI and age")
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
plt.ylabel("BMI")
plt.xlabel("Age")
plt.show()

# Calculating feature importance for ranges of values

In [ ]:
#creating a new dataframe to calculate feature importance
plot_these = ["Pregnancies", "Age", "Glucose", "Insulin", "BloodPressure", "SkinThickness", "BMI", "DiabetesPedigreeFunction" ]
df["Pregnancies"] = pd.cut(df["Pregnancies"], 5, labels=range(0, 5))
df["Age"] = pd.cut(df["Age"], 5, labels=range(0, 5))
df["Glucose"] = pd.cut(df["Glucose"], 5, labels=range(0, 5))
df["Insulin"] = pd.cut(df["Insulin"], 10, labels=range(0, 10))
df["BloodPressure"] = pd.cut(df["BloodPressure"], 5, labels=range(0, 5))
df["SkinThickness"] = pd.cut(df["SkinThickness"], 5, labels=range(0, 5))
df["BMI"] = pd.cut(df["BMI"], 5, labels=range(30, 35))
df[ "DiabetesPedigreeFunction"] = pd.cut(df[ "DiabetesPedigreeFunction" ], 5, labels=range(0, 5))

a = pd.get_dummies(df, columns=plot_these, 
                   prefix=plot_these, drop_first=True)

a.head()

The continuous values are replaced with discrete subcategories, to show the importance of different ranges in the whole domain. The feature importance was calculated by random forest classifier. From this graph it seems that the glucose level, BMI and Skint Thickness have the highest predictive power.

In [ ]:
a.drop("Test positivity", axis=1, inplace=True)

rf = RandomForestClassifier()
rf.fit(a.drop("Outcome", axis=1), a.Outcome)
importances = rf.feature_importances_
features = pd.Series(importances, index=a.drop("Outcome", axis=1).columns)
plt.figure(figsize=(12, 16))
features.plot(kind="barh")
plt.show()

# Creating a machine learning modell

Here we apply XGBoost classifier to Outcome. We also remove the two least significant categories (SkinThickness 3 and 4).

In [ ]:
X=a.drop(["Outcome", "SkinThickness_3", "SkinThickness_4"], axis=1)
y = a.Outcome
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# A small illustrative grid so this runs in a few seconds. A real search widens these
# ranges (more depths, estimators, learning rates) -- that is what "takes forever".
xgb = XGBClassifier(objective="binary:logistic", eval_metric="auc")
params = {"max_depth": [1, 3, 5],
          "n_estimators": [100, 300],
          "learning_rate": [0.01, 0.05, 0.1]}
grid_xgb = GridSearchCV(estimator=xgb, param_grid=params, cv=5, n_jobs=-1)
grid_xgb.fit(X_train, y_train)
xgb_pred = grid_xgb.predict(X_test)
print(classification_report(y_test, xgb_pred))

In [ ]:
print(grid_xgb.best_params_)
print('\n')
# best_score_ is the mean CV score of the BEST params -- not the average over all
# param combinations (which would mix good and bad settings together).
print("best CV score:", grid_xgb.best_score_)
print('\n')
print(classification_report(y_test, xgb_pred))

In [ ]:
sns.set_context("talk")
plt.figure(figsize=(12, 9))
sns.heatmap(confusion_matrix(y_test, xgb_pred), annot=True, xticklabels=["Healthy", "Sick"], yticklabels=["Healthy", "Sick"], fmt="g", cmap="icefire_r")
plt.ylabel("Predicted")
plt.xlabel("Actual")
plt.title("Confusion matrix of xGBoosting")
plt.show()

In [ ]:


xgb = XGBClassifier(objective="binary:logistic", eval_metric="auc", max_depth=1,
                    n_estimators=500,learning_rate=0.05,colsample_bytree=0.35,subsample=0.5)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)


xgb_prob = xgb.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, xgb_prob)
sns.set_style("darkgrid")
sns.set_context("poster")
plt.figure(figsize=(15, 10))
plt.plot([0, 1], [0, 1], 'k--')
sns.lineplot(x=fpr, y=tpr, alpha=0.6)
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curve')
plt.legend(["Baseline", "xGBoosting"])
plt.show()
print(roc_auc_score(y_test, xgb_prob))